# Lab 8: Check the claims
Four sentences that someone could write about the Airbnb table. For each one I test the number, look at what is behind it, decide a verdict, and write a new sentence I could defend. For every claim I ask: **what does the sentence not say?**

In [ ]:
import pandas as pd
import numpy as np

# change the list if your file has other date columns
listings = pd.read_csv('data/listings_clean.csv',
                       parse_dates=['first_review', 'last_review', 'last_scraped'])
if 'has_reviews' not in listings:
    listings['has_reviews'] = listings['number_of_reviews'] > 0
print(listings.shape)

## Claim 0 (the example): "The typical Stockholm listing has 36 reviews."

In [ ]:
reviews = listings['number_of_reviews']
print(reviews.mean().round(1), reviews.median())                    # 35.8 9.0
print((reviews == 0).sum(), ((reviews == 0).mean() * 100).round(1))  # 644 14.1
print((reviews > 100).sum())                                         # 426

**Verdict:** Does not hold. The number 36 is right, but only as a mean. "Typical" means the median, and that is 9.

**New sentence:** "The median Stockholm listing has 9 reviews, and 14 percent have none. The mean is 36, because a few hundred listings have more than 100 reviews." 

## Claim 1: "A night in Stockholm costs 1,791."

In [ ]:
price = listings['price']
print('Median price:', price.median())                                     # 1791.0
print('Listings with a price:', price.notna().sum())                       # 2957
print('Listings without a price:', price.isna().sum(), round(price.isna().mean() * 100, 1), '%')   # 1617, 35.4 %

In [ ]:
# What do the listings without a price have in common?
no_price = listings[price.isna()]
print('No price and no open days (availability_365 == 0):', (no_price['availability_365'] == 0).sum())   # 1490 of 1617
print('Has a price and no open days:', (listings[price.notna()]['availability_365'] == 0).sum())         # 4

In [ ]:
# Long stays against the others (only listings with a price)
with_price = listings[price.notna()]
long_stay = with_price['minimum_nights'] >= 28
print(long_stay.sum(), with_price[long_stay]['price'].median().round(1))        # 97 listings, median 708.9
print((~long_stay).sum(), with_price[~long_stay]['price'].median().round(1))    # 2860 listings, median 1839.2

**What is behind the number:** 1791 is correct, but it is the median of only 2957 listings. 1617 listings (35.4 %) have no price, and 1490 of those have no open days, so they probably cannot be booked at the moment. 97 listings with a price need 28 nights or more, and their median price is only 708.9, against 1839.2 for the other 2860. The sentence also does not say which currency or which date. (The data dictionary says the price is in the local currency, which is SEK in Stockholm.)

**Verdict:** Holds with reservations.

**New sentence:** "On 26 and 27 September 2026 the median price per night was 1,791 SEK for the 2,957 Airbnb listings in Stockholm that had a price, which is 65 percent of all listings." 

## Claim 2: "The listings with a perfect 5.00 rating are the best in Stockholm."

In [ ]:
rated = listings[listings['has_reviews']]            # only listings with a rating can have a perfect one
print('Rated listings:', len(rated))                                                 # 3930
perfect = rated[rated['review_scores_rating'] == 5.0]
others = rated[rated['review_scores_rating'] < 5.0]
print('Perfect 5.00:', len(perfect), round(len(perfect) / len(rated) * 100, 1), '%')     # 1484, 37.8 %

In [ ]:
many = perfect[perfect['number_of_reviews'] >= 20]
print('Perfect with at least 20 reviews:', len(many), round(len(many) / len(perfect) * 100, 1), '%')   # 130, 8.8 %
print('Median reviews, perfect:', perfect['number_of_reviews'].median())             # 4.0
print('Median reviews, the others:', others['number_of_reviews'].median())           # 25.0

**What is behind the number:** 3930 listings have a rating, and 1484 of them (37.8 %) have a perfect 5.00. But only 130 of the perfect ones have at least 20 reviews (8.8 %), so 91 % of them have fewer than 20. The median number of reviews is 4 for the perfect listings and 25 for the others. A perfect score is mostly a sign of **few reviews**: with 1 to 4 guests it is easy to get all fives, with 100 guests it is not. The sentence also leaves out that "best" is not measured by the rating alone.

**Verdict:** Does not hold.

**New sentence:** "38 percent of the rated listings have a perfect 5.00, but 91 percent of them have fewer than 20 reviews, so the rating says little about which are best." 

## Claim 3: "The market has grown: only 131 listings started in 2016, but 610 in 2024."

In [ ]:
first_year = listings['first_review'].dt.year
print(first_year.value_counts().sort_index().loc[[2016, 2024]])       # 2016: 131, 2024: 610

In [ ]:
# Which of them were reviewed recently (in 2026)?
listings['first_year'] = first_year
listings['reviewed_2026'] = listings['last_review'].dt.year == 2026
listings.groupby('first_year')['reviewed_2026'].mean().round(2).loc[[2016, 2024]]
# 2016: 0.53, 2024: 0.67

**What is behind the number:** The counts are correct (131 and 610), but the table only holds listings that **exist today**. Listings that started in 2016 and have closed since are not in it, so the old years look smaller than they really were. Also, `first_review` is the first review, not the day the listing started. Among the 2016 listings only 53 % were reviewed in 2026, against 67 % for the 2024 listings, which fits a picture where many old listings have stopped. The table cannot show how the market grew.

**Verdict:** Cannot be concluded from this table.

**New sentence:** "Of the listings that still exist, 131 got their first review in 2016 and 610 in 2024. We cannot say how many were started in those years." 

## Claim 4: "Most hosts in Stockholm are professional companies with many listings."

In [ ]:
per_host = listings.groupby('host_id').size()
print('Listings:', len(listings), ' Hosts:', listings['host_id'].nunique())        # 4574, 3591
print('Hosts with one listing:', (per_host == 1).sum(), round((per_host == 1).mean() * 100, 1), '%')   # 3237, 90.1 %

In [ ]:
several = per_host[per_host >= 2]
print('Hosts with two or more:', len(several), round(len(several) / len(per_host) * 100, 1), '%')     # 354, 9.9 %
print('They own', several.sum(), 'listings =', round(several.sum() / len(listings) * 100, 1), '% of all listings')   # 1337, 29.2 %
print('The largest host has', per_host.max(), 'listings')                          # 49

**What is behind the number:** A listing is not a host. The table has 4574 listings but only 3591 hosts. 3237 hosts (90.1 %) have just one listing. 354 hosts (9.9 %) have two or more, and together they own 1337 listings, 29.2 % of all listings. The biggest host has 49. So "most hosts have many listings" is wrong. Also, there is no column that says whether a host is a company, so "professional companies" cannot be checked at all.

**Verdict:** Does not hold (and the word "company" cannot be concluded from this table).

**New sentence:** "90 percent of the hosts have only one listing, but the 10 percent with two or more own 29 percent of all listings." 

## Task 5: Data note (no verdict)

In [ ]:
# numbers for the note
print('Prices missing:', round(listings['price'].isna().mean() * 100, 1), '%')                    # 35.4
print('Ratings missing:', round(listings['review_scores_rating'].isna().mean() * 100, 1), '%')     # 14.1
print([c for c in listings.columns if c.startswith('estimated_')])        # calculated columns
print(listings['last_scraped'].dt.date.value_counts())                    # 26 and 27 September 2026

**Data note**

This table is a snapshot of 4,574 Airbnb listings in Stockholm, collected on 26 and 27 September 2026 by Inside Airbnb, and it may be used with credit to insideairbnb.com (licence CC BY 4.0). Many values are missing: 35 percent of the prices (mostly listings with no open days) and 14 percent of the ratings (the listings without any review), and 15 columns in the raw file have no values at all. Listings that have closed are not in the table, and neither are hotels or other platforms, so it shows what exists on one day and not how the market has changed. Some columns are calculated and not measured, for example the `estimated_` columns for occupancy and revenue, which are Inside Airbnb's own estimates. The table cannot tell us why a price is what it is, or whether a guest was satisfied, and a listing is not a host. If I read one of the four claims in a newspaper, I would most want to correct claim 2, because 91 percent of the perfect 5.00 listings have fewer than 20 reviews, and a reader could choose a place from a number that says very little.